### Notebook Purposes
```text 
01_dataset_exploration
        ↓
   What data do we have?

02_training_data_validation
        ↓
   Can we correctly turn games into training observations?

03_train_policy_v1
        ↓
   Can we train a model and produce policy_v1.pt?

04_evaluate_policy_v1
        ↓
   Can we load policy_v1.pt and make useful predictions?
```

### Inference Process
* FEN is the board as the input
* MovePrediction is the decoded value as the output
    - MovePrediction(from_square='e2', to_square='e4', promotion=None, score=0.4525875449180603, model_version='policy-v1')
```
FEN
 ↓
chess.Board
 ↓
encode_board
 ↓
[18, 8, 8]
 ↓
unsqueeze(0)
 ↓
[1, 18, 8, 8]
 ↓
PolicyCNN
 ↓
[1, 20,480] logits
 ↓
argmax
 ↓
class ID
 ↓
decode_move
 ↓
MovePrediction
```

```
MODEL_PATH
"artifacts/policy_v1.pt"
        │
        ▼
PolicyPredictor.__init__()
        │
        ├── choose device
        │      └── CUDA on your machine
        │
        ├── PolicyCNN()
        │      └── create fresh CNN architecture
        │
        ├── torch.load(policy_v1.pt)
        │      └── load epoch-7 state_dict
        │
        ├── model.load_state_dict(...)
        │      └── put learned weights into CNN
        │
        └── model.eval()
               └── inference mode
```

In [1]:
from pathlib import Path

import chess
from services.inference import ONNXPolicyPredictor


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()

    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists() and (
            candidate / "artifacts"
        ).exists():
            return candidate

    raise FileNotFoundError("Could not locate the repository root")


REPO_ROOT = find_repo_root()

MODEL_ONNX_PATH = REPO_ROOT / "artifacts" / "notebook" / "policy_v1.onnx"
MODEL_METADATA_PATH = REPO_ROOT / "artifacts" / "notebook" / "policy_v1.json"

predictor = ONNXPolicyPredictor(
    MODEL_ONNX_PATH,
    MODEL_METADATA_PATH,
)

In [3]:
print(predictor.model_version)
print(predictor.onnx_path)
print(type(predictor.session))
print(predictor.session.get_providers())

policy-v1
/mnt/c/working/public/chess-multiplayer/artifacts/notebook/policy_v1.onnx
<class 'onnxruntime.capi.onnxruntime_inference_collection.InferenceSession'>
['CPUExecutionProvider']


In [6]:
for model_input in predictor.session.get_inputs():
    print(
        model_input.name,
        model_input.shape,
        model_input.type,
    )

for model_output in predictor.session.get_outputs():
    print(
        model_output.name,
        model_output.shape,
        model_output.type,
    )

board ['batch', 18, 8, 8] tensor(float)
logits ['batch', 20480] tensor(float)


In [4]:
# Starting-position prediction

fen = chess.STARTING_FEN

prediction = predictor.predict(fen)

print(f"FEN: {fen}")
print(f"Prediction: {prediction}")

FEN: rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR w KQkq - 0 1
Prediction: MovePrediction(from_square='e2', to_square='e4', promotion=None, score=0.5448753237724304, model_version='policy-v1')


In [5]:
# Standalone inference smoke tests

after_e4 = chess.Board()
after_e4.push_uci("e2e4")

after_e4_e5 = chess.Board()
after_e4_e5.push_uci("e2e4")
after_e4_e5.push_uci("e7e5")

positions = {
    "Starting position": chess.STARTING_FEN,
    "After e4": after_e4.fen(),
    "After e4 e5": after_e4_e5.fen(),
    "Middlegame": (
        "r1bq1rk1/pp2bppp/2n1pn2/2pp4/3P4/2PBPN2/PP1NBPPP/R2Q1RK1 w - - 2 9"
    ),
    "Endgame": ("8/5pk1/6p1/8/4P3/5K2/8/8 w - - 0 1"),
}

for name, fen in positions.items():
    board = chess.Board(fen)

    prediction = predictor.predict(fen)

    move_uci = prediction.from_square + prediction.to_square

    if prediction.promotion is not None:
        move_uci += prediction.promotion[0]

    move = chess.Move.from_uci(move_uci)

    print(name)
    print(f"  Prediction: {move_uci}")
    print(f"  Score:      {prediction.score:.4f}")
    print(f"  Legal:      {move in board.legal_moves}")
    print()

Starting position
  Prediction: e2e4
  Score:      0.5449
  Legal:      True

After e4
  Prediction: d7d5
  Score:      0.2592
  Legal:      True

After e4 e5
  Prediction: g1f3
  Score:      0.4817
  Legal:      True

Middlegame
  Prediction: d1c2
  Score:      0.2265
  Legal:      True

Endgame
  Prediction: f3e3
  Score:      0.3118
  Legal:      True

